# WakeRef — Convertir le modèle JIB en ONNX + pousser sur Hugging Face

À faire APRÈS l'entraînement jib. Tu auras besoin :
- du fichier **`whisper-wakeref-jib.zip`** (le modèle),
- d'un compte **huggingface.co** + un **token "Write"** (Settings → Access Tokens).

Lance les cellules dans l'ordre (▶). Le GPU n'est pas nécessaire ici.

## 1. Installer Optimum (l'outil de conversion ONNX) — ~2 min

In [ ]:
!pip -q install "optimum[onnxruntime]"
print('OK')

## 2. Récupérer le modèle (`whisper-wakeref-jib.zip`) depuis Google Drive
Mets ton zip du modèle dans Google Drive, puis renseigne `ZIP` ci-dessous. Drive ↔ Colab = transfert **Google-interne (rapide)**, contrairement à `files.upload()` (bridé). Repli « upload manuel » en commentaire.

In [ ]:
# Modèle depuis Google Drive (rapide) — plutôt que files.upload() (bridé).
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/whisper-wakeref-jib.zip'   # <- adapte le nom/chemin

# Repli (upload depuis ta machine) : décommente les 2 lignes ci-dessous.
# from google.colab import files
# ZIP = '/content/' + next(iter(files.upload()))

!unzip -q -o "$ZIP" -d model_pt
import os
PT = next(r for r,_,fs in os.walk('model_pt') if 'config.json' in fs)
print('modele PyTorch :', PT)

## 3. Convertir en ONNX (~2-3 min ; whisper-small = plus gros que base)

In [ ]:
!optimum-cli export onnx -m "$PT" --task automatic-speech-recognition-with-past onnx_out
import os; print('produit :', os.listdir('onnx_out'))

## 4. Réorganiser au format de l'app (+ copier le tokenizer)
Configs + tokenizer à la racine, `.onnx` dans `onnx/`. Optimum n'exporte PAS le
tokenizer → on le reprend du modèle PyTorch (sinon l'app plante : `tokenizer is null`).
Le tokenizer d'un whisper-small fine-tuné = celui de whisper-small (vocabulaire inchangé).

In [ ]:
import os, glob, shutil
os.makedirs('hf/onnx', exist_ok=True)
# .onnx -> hf/onnx/
for f in glob.glob('onnx_out/*.onnx') + glob.glob('onnx_out/*.onnx_data'):
    shutil.move(f, 'hf/onnx/' + os.path.basename(f))
# configs produits par Optimum -> hf/
for f in glob.glob('onnx_out/*'):
    if os.path.isfile(f): shutil.copy(f, 'hf/' + os.path.basename(f))
# tokenizer depuis le modele PyTorch -> hf/
for name in ['tokenizer.json','tokenizer_config.json','vocab.json','merges.txt',
             'normalizer.json','special_tokens_map.json','added_tokens.json']:
    src = os.path.join(PT, name)
    if os.path.exists(src): shutil.copy(src, 'hf/' + name)
OUT = 'hf'
print('racine :', sorted(os.listdir('hf')))
print('onnx/  :', sorted(os.listdir('hf/onnx')))
assert 'tokenizer.json' in os.listdir('hf'), 'tokenizer manquant !'

## 4bis. Quantizer en q8 (÷4 la taille : ~855 Mo → ~215 Mo)
Poids fp32 → int8 (dynamique, poids seuls → qualité quasi intacte sur un modèle à domaine fermé). transformers.js charge cette variante via `dtype: 'q8'` (suffixe attendu : `_quantized`). On **garde aussi le fp32** : l'app choisit le dtype, le navigateur ne télécharge que celui demandé.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import os
# encodeur + décodeur fusionné → variantes int8 (_quantized) à côté des fp32.
for base in ['encoder_model', 'decoder_model_merged']:
    src = f'hf/onnx/{base}.onnx'
    if not os.path.exists(src):
        print('absent, ignore :', src); continue
    dst = f'hf/onnx/{base}_quantized.onnx'
    quantize_dynamic(src, dst, weight_type=QuantType.QInt8)
    print(f'{base}: {os.path.getsize(src)/1e6:.0f} Mo fp32 -> {os.path.getsize(dst)/1e6:.0f} Mo q8')
print('onnx/ :', sorted(os.listdir('hf/onnx')))

## 5. Pousser sur Hugging Face
Remplis **TON_USER** et **ton token Write** (entre les guillemets), puis lance.
Le token en clair : ne partage pas le notebook, ou efface-le après.

In [ ]:
USER  = 'TON_USER'                       # <- ton identifiant huggingface.co
TOKEN = 'hf_colle_ton_token_Write'       # <- huggingface.co/settings/tokens (Write)
REPO  = f'{USER}/whisper-wakeref-jib-onnx'
from huggingface_hub import login, HfApi
login(token=TOKEN)                        # authentifie AVANT le push (bloquant)
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO)
print('OK pousse :', f'https://huggingface.co/{REPO}')

## C'est fait
Modèle complet (onnx **fp32** + **q8** + tokenizer) sur `https://huggingface.co/TON_USER/whisper-wakeref-jib-onnx`.

L'app est déjà branchée (WebGPU → fp32, WASM → q8). **Vide le cache du modèle** (DevTools → Application → Clear site data) puis recharge `/judge/voix`, **dictée libre (jib)**, **Précharger**, teste la vitesse.